# Textual Contextual Bandits with Moderation Data


---

## Problem setting

You are supposed to design a moderation decision policy for user-generated text. For each post, the system must choose one action:

| Action id | Action | Operational meaning |
|---:|---|---|
| 0 | `allow` | publish immediately |
| 1 | `review` | route to human moderation |
| 2 | `block` | automatically remove |

You only observe the reward of the action you chose. This is contextual bandit feedback, not supervised full-information learning.

---
## Important note on running

This notebook requires internet access the first time it downloads the Hugging Face datasets. After caching, it can be rerun offline on the same machine.

Install dependencies:

```bash
pip install numpy pandas matplotlib torch datasets tqdm
```

In [ ]:
pip install numpy pandas matplotlib torch datasets tqdm

In [ ]:
import math
import re
import hashlib
import random
import warnings
from dataclasses import dataclass
from typing import Callable, Dict, Iterable, List, Optional, Tuple
from pathlib import Path
import urllib.request
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# 1. local CSV cache,
# 2. official TweetEval GitHub raw files,
# 3. Hugging Face datasets,
# 4. original-repo prepared memmaps if available.
try:
    from datasets import load_dataset
except Exception:
    load_dataset = None

SEED = 17
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

ACTIONS = ["allow", "review", "block"]
ALLOW, REVIEW, BLOCK = 0, 1, 2
N_ACTIONS = len(ACTIONS)

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
@dataclass
class Config:
    base_dataset_name: str = "cardiffnlp/tweet_eval"
    base_dataset_config: str = "offensive"

    ood_dataset_name: str = "cardiffnlp/tweet_eval"
    ood_dataset_config: str = "hate"

    # Keep these modest for CPU labs. Increase for stronger results.
    n_logged: int = 6000
    n_eval: int = 2000
    n_ood: int = 2000

    # Hashing vectorizer
    hash_dim: int = 512
    min_token_len: int = 2

    # Neural reward model
    hidden: int = 128
    dropout_p: float = 0.20
    batch_size: int = 128
    reward_epochs: int = 8
    ensemble_epochs: int = 5
    risk_epochs: int = 8
    lr: float = 1e-3
    weight_decay: float = 1e-4

    # Logging policy
    behavior_temperature: float = 0.70
    behavior_uniform_mix: float = 0.12

    # Policies
    epsilon: float = 0.10
    ensemble_size: int = 3
    mc_samples: int = 15

    # Safe exploration
    unsafe_threshold: float = 0.55
    extreme_unsafe_threshold: float = 0.80
    safety_margin: float = 0.10

    # LinUCB and NeuralUCB
    linucb_alpha: float = 0.45
    linucb_l2: float = 1.0
    neuralucb_alpha: float = 0.70
    neural_feature_dim: int = 128

cfg = Config()
cfg


## 1. Real dataset loading
### Task 1

Run the loader and inspect examples.

In [ ]:
CACHE_DIR = Path("./tweeteval_cache")
CACHE_DIR.mkdir(exist_ok=True)

TWEETEVAL_GITHUB_BASE = "https://raw.githubusercontent.com/cardiffnlp/tweeteval/main/datasets"

# GitHub uses val, while Hugging Face commonly calls this validation.
SPLIT_TO_GITHUB = {
    "train": "train",
    "validation": "val",
    "val": "val",
    "test": "test",
}

def _sample_or_all(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if len(df) <= n:
        return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return df.sample(n=n, random_state=seed).reset_index(drop=True)

def _read_cached_csv(task: str, split: str) -> Optional[pd.DataFrame]:
    path = CACHE_DIR / f"tweeteval_{task}_{split}.csv"
    if path.exists():
        df = pd.read_csv(path)
        if {"text", "label"}.issubset(df.columns):
            df["label"] = df["label"].astype(int)
            df["source"] = f"local_cache/{path.name}"
            return df
    return None

def _write_cached_csv(df: pd.DataFrame, task: str, split: str) -> None:
    path = CACHE_DIR / f"tweeteval_{task}_{split}.csv"
    df[["text", "label", "source"]].to_csv(path, index=False)

def _download_text(url: str, timeout: int = 30, retries: int = 4) -> str:
    last_error = None
    for attempt in range(retries):
        try:
            req = urllib.request.Request(
                url,
                headers={
                    "User-Agent": "Mozilla/5.0 educational-notebook",
                    "Accept": "text/plain,*/*",
                },
            )
            with urllib.request.urlopen(req, timeout=timeout) as response:
                return response.read().decode("utf-8", errors="replace")
        except Exception as e:
            last_error = e
            sleep_s = min(2 ** attempt, 8)
            print(f"Download failed attempt {attempt+1}/{retries}: {url}\n  {type(e).__name__}: {e}\n  retrying in {sleep_s}s")
            time.sleep(sleep_s)
    raise RuntimeError(f"Could not download {url}: {last_error}")

def load_tweeteval_from_github(task: str, split: str) -> pd.DataFrame:
    """Load official TweetEval text/label files directly from GitHub raw.
    Labels are stored as a single whitespace-separated line in the official repo.
    Text files contain one tweet per line.
    """
    cached = _read_cached_csv(task, split)
    if cached is not None:
        return cached

    gh_split = SPLIT_TO_GITHUB[split]
    text_url = f"{TWEETEVAL_GITHUB_BASE}/{task}/{gh_split}_text.txt"
    label_url = f"{TWEETEVAL_GITHUB_BASE}/{task}/{gh_split}_labels.txt"

    print(f"Downloading TweetEval {task}/{split} from official GitHub raw files.")
    text_raw = _download_text(text_url)
    label_raw = _download_text(label_url)

    texts = [line.strip() for line in text_raw.splitlines()]
    labels = [int(x) for x in label_raw.split()]

    if len(texts) != len(labels):
        raise ValueError(
            f"Text/label length mismatch for {task}/{split}: "
            f"{len(texts)} texts vs {len(labels)} labels"
        )

    df = pd.DataFrame({"text": texts, "label": labels})
    df["label"] = df["label"].astype(int)
    df["source"] = f"github_raw/tweeteval/{task}/{split}"
    _write_cached_csv(df, task, split)
    return df

def load_tweeteval_from_hf(dataset_name: str, config_name: str, split: str) -> pd.DataFrame:
    if load_dataset is None:
        raise RuntimeError("datasets is not installed; skipping Hugging Face fallback.")
    ds = load_dataset(dataset_name, config_name, split=split)
    df = pd.DataFrame(ds)
    if "text" not in df.columns or "label" not in df.columns:
        raise ValueError(f"Expected text and label columns, got {df.columns.tolist()}")
    df = df[["text", "label"]].copy()
    df["label"] = df["label"].astype(int)
    df["source"] = f"huggingface/{dataset_name}/{config_name}/{split}"
    return df

def load_original_repo_memmap(task: str, split_hint: str = "train") -> pd.DataFrame:
    """Fallback for users who already ran the original repo's prepare_data.py.

    The original repo stores GPT-2 token ids and labels in hf_data/tweet_eval/{task}.
    This fallback creates pseudo-text from token ids so the rest of the notebook can
    still run without Hugging Face/GitHub. Prefer GitHub/HF because they preserve text.
    """
    possible_roots = [
        Path("hf_data") / "tweet_eval" / task,
        Path("./llm-bandit-main") / "hf_data" / "tweet_eval" / task,
        Path("../llm-bandit-main") / "hf_data" / "tweet_eval" / task,
        Path("/mnt/data/llm-bandit-main") / "hf_data" / "tweet_eval" / task,
    ]
    root = next((p for p in possible_roots if (p / "data_labels.bin").exists()), None)
    if root is None:
        raise FileNotFoundError(f"No original repo memmap found for task={task}")

    block_size = 64
    labels = np.memmap(root / "data_labels.bin", dtype=np.uint16, mode="r")
    text_ids = np.memmap(root / "data_text.bin", dtype=np.uint16, mode="r").reshape(-1, block_size)

    pseudo_texts = []
    for row in text_ids:
        toks = [f"tok_{int(x)}" for x in row if int(x) != 0]
        pseudo_texts.append(" ".join(toks[:80]))

    df = pd.DataFrame({"text": pseudo_texts, "label": labels.astype(int)})
    df["source"] = f"original_repo_memmap/{root}"
    return df

def load_real_tweeteval_task(task: str, split: str, cfg: Config) -> pd.DataFrame:
    """Robust loader that avoids Hugging Face unless needed."""
    try:
        return load_tweeteval_from_github(task, split)
    except Exception as github_error:
        print(f"GitHub raw loading failed for {task}/{split}: {type(github_error).__name__}: {github_error}")

    try:
        return load_tweeteval_from_hf(cfg.base_dataset_name, task, split)
    except Exception as hf_error:
        print(f"Hugging Face fallback failed for {task}/{split}: {type(hf_error).__name__}: {hf_error}")

    try:
        return load_original_repo_memmap(task, split)
    except Exception as memmap_error:
        raise RuntimeError(
            f"All real-data loading methods failed for TweetEval {task}/{split}.\n"
            f"Fix options:\n"
            f"  1. Re-run later; Hugging Face 429 usually clears.\n"
            f"  2. Ensure GitHub raw access is available.\n"
            f"  3. Manually place CSVs in ./tweeteval_cache/ named tweeteval_{task}_{split}.csv "
            f"with columns text,label.\n"
            f"  4. Run the original repo prepare_data.py --data {task} to create hf_data memmaps.\n"
            f"Last memmap error: {type(memmap_error).__name__}: {memmap_error}"
        )

def load_real_moderation_data(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load real in-domain and out-of-domain moderation datasets robustly."""
    # In-domain offensive data.
    train = load_real_tweeteval_task("offensive", "train", cfg)
    validation = load_real_tweeteval_task("offensive", "validation", cfg)
    test = load_real_tweeteval_task("offensive", "test", cfg)
    eval_pool = pd.concat([validation, test], ignore_index=True)

    # OOD hate data.
    ood_validation = load_real_tweeteval_task("hate", "validation", cfg)
    ood_test = load_real_tweeteval_task("hate", "test", cfg)
    ood_pool = pd.concat([ood_validation, ood_test], ignore_index=True)

    logged = _sample_or_all(train, cfg.n_logged, SEED + 1)
    eval_df = _sample_or_all(eval_pool, cfg.n_eval, SEED + 2)
    ood_df = _sample_or_all(ood_pool, cfg.n_ood, SEED + 3)

    return logged, eval_df, ood_df

logged_raw, eval_raw, ood_raw = load_real_moderation_data(cfg)

print("Loaded real datasets:")
display(pd.DataFrame({
    "split": ["logged_train", "in_domain_eval", "ood_hate_eval"],
    "rows": [len(logged_raw), len(eval_raw), len(ood_raw)],
    "harmful_rate": [logged_raw.label.mean(), eval_raw.label.mean(), ood_raw.label.mean()],
    "source": [logged_raw.source.iloc[0], eval_raw.source.iloc[0], ood_raw.source.iloc[0]],
}))

display(logged_raw.sample(min(10, len(logged_raw)), random_state=SEED))

## 2. Text diagnostics and severity proxy

 Real moderation costs often depend on severity, but `tweet_eval/offensive` only provides binary labels.

We therefore construct a **severity proxy** from text signals and the harmful label. This is not perfect and should be discussed critically. It is used only to define richer reward and regret for the bandit environment.

In [ ]:
TOKEN_RE = re.compile(r"[A-Za-z_@#!']+")

MILD_RISK_TERMS = {
    "stupid", "idiot", "shut", "hate", "trash", "dumb", "fool", "sucks",
    "worst", "clown", "garbage", "nonsense", "awful", "pathetic"
}

def tokenize(text: str) -> List[str]:
    return TOKEN_RE.findall(str(text).lower())

def text_risk_score(text: str) -> float:
    text = str(text)
    toks = tokenize(text)
    if not toks:
        return 0.0

    clean = [t.strip("_@#!'") for t in toks]
    risk_terms = sum(t in MILD_RISK_TERMS for t in clean)

    term_score = 1.0 - math.exp(-0.9 * risk_terms)
    exclaim_score = min(text.count("!") / 4.0, 1.0)

    alpha_words = re.findall(r"\b[A-Z]{2,}\b", text)
    word_count = max(1, len(re.findall(r"\b[A-Za-z]+\b", text)))
    caps_score = min(len(alpha_words) / word_count * 4.0, 1.0)

    hashtag_score = min(sum(tok.startswith("#") for tok in toks) / 3.0, 1.0)

    score = (
        0.62 * term_score
        + 0.16 * exclaim_score
        + 0.14 * caps_score
        + 0.08 * hashtag_score
    )
    return float(np.clip(score, 0.0, 1.0))

def severity_proxy(text: str, label: int) -> float:
    r = text_risk_score(text)
    if int(label) == 1:
        severity = 0.45 + 0.55 * r
    else:
        severity = 0.20 * r
    return float(np.clip(severity, 0.0, 1.0))

def add_text_diagnostics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    toks = out["text"].map(tokenize)

    out["n_tokens"] = toks.map(len)
    out["risk_score"] = out["text"].map(text_risk_score)
    out["severity"] = [
        severity_proxy(text, label)
        for text, label in zip(out["text"].values, out["label"].values)
    ]

    out["has_risk_term"] = toks.map(
        lambda ts: any(t.strip("_@#!'") in MILD_RISK_TERMS for t in ts)
    )
    out["has_mention"] = toks.map(lambda ts: any(t.startswith("@") for t in ts))
    out["has_hashtag"] = toks.map(lambda ts: any(t.startswith("#") for t in ts))
    out["n_exclamations"] = out["text"].astype(str).str.count("!")
    out["high_text_risk"] = out["risk_score"] >= 0.50
    return out

logged_df = add_text_diagnostics(logged_raw)
eval_df = add_text_diagnostics(eval_raw)
ood_df = add_text_diagnostics(ood_raw)

display(pd.DataFrame({
    "split": ["logged", "in_domain_eval", "ood_hate_eval"],
    "harmful_rate": [logged_df.label.mean(), eval_df.label.mean(), ood_df.label.mean()],
    "mean_risk_score": [logged_df.risk_score.mean(), eval_df.risk_score.mean(), ood_df.risk_score.mean()],
    "mean_severity": [logged_df.severity.mean(), eval_df.severity.mean(), ood_df.severity.mean()],
}))


## 3. Reward, oracle, regret, and cost-sensitivity analysis

A moderation classifier predicts a label. A contextual bandit chooses an action.

This section defines three product cost profiles:

1. `balanced`: balanced platform policy.
2. `high_safety`: strict platform; allowing harmful content is extremely costly.
3. `free_expression`: blocking benign content is especially costly.

You must evaluate how the optimal action distribution changes under different cost profiles.

### Task 3

Implement:

- `expected_reward_for_action`
- `reward_table`
- `oracle_action_and_value`
- `expected_regret`
- cost-profile comparison


In [ ]:
COST_PROFILES = {
    "balanced": {
        "benign_allow": 1.00,
        "benign_review": 0.30,
        "benign_block": -1.60,
        "harm_allow_base": -2.00,
        "harm_allow_severity": -2.20,
        "harm_review_base": 0.05,
        "harm_review_severity": 0.20,
        "harm_block_base": 0.75,
        "harm_block_severity": 0.55,
    },
    "high_safety": {
        "benign_allow": 0.85,
        "benign_review": 0.20,
        "benign_block": -1.10,
        "harm_allow_base": -3.20,
        "harm_allow_severity": -3.50,
        "harm_review_base": 0.20,
        "harm_review_severity": 0.30,
        "harm_block_base": 1.00,
        "harm_block_severity": 0.85,
    },
    "free_expression": {
        "benign_allow": 1.15,
        "benign_review": 0.15,
        "benign_block": -2.80,
        "harm_allow_base": -1.20,
        "harm_allow_severity": -1.40,
        "harm_review_base": 0.05,
        "harm_review_severity": 0.15,
        "harm_block_base": 0.45,
        "harm_block_severity": 0.35,
    },
}

ACTIVE_PROFILE = "balanced"

def expected_reward_for_action(label: int, severity: float, action: int, profile: str = ACTIVE_PROFILE) -> float:
    p = COST_PROFILES[profile]
    harmful = int(label) == 1
    severity = float(np.clip(severity, 0.0, 1.0))

    if not harmful:
        if action == ALLOW:
            return float(p["benign_allow"])
        if action == REVIEW:
            return float(p["benign_review"])
        if action == BLOCK:
            return float(p["benign_block"])
    else:
        if action == ALLOW:
            return float(p["harm_allow_base"] + p["harm_allow_severity"] * severity)
        if action == REVIEW:
            return float(p["harm_review_base"] + p["harm_review_severity"] * severity)
        if action == BLOCK:
            return float(p["harm_block_base"] + p["harm_block_severity"] * severity)

    raise ValueError(f"Unknown action id: {action}")


def sample_observed_reward(
    label: int,
    severity: float,
    action: int,
    rng: np.random.Generator,
    profile: str = ACTIVE_PROFILE,
    noise_std: float = 0.08,
) -> float:
    return float(expected_reward_for_action(label, severity, action, profile) + rng.normal(0.0, noise_std))

def reward_table(df: pd.DataFrame, profile: str = ACTIVE_PROFILE) -> np.ndarray:
    rewards = np.zeros((len(df), N_ACTIONS), dtype=np.float32)
    labels = df["label"].values
    severities = df["severity"].values
    for a in range(N_ACTIONS):
        rewards[:, a] = [
            expected_reward_for_action(label, sev, a, profile)
            for label, sev in zip(labels, severities)
        ]
    return rewards

def oracle_action_and_value(df: pd.DataFrame, profile: str = ACTIVE_PROFILE) -> Tuple[np.ndarray, np.ndarray]:
    rewards = reward_table(df, profile)
    actions = rewards.argmax(axis=1).astype(int)
    values = rewards[np.arange(len(df)), actions]
    return actions, values

def expected_regret(df: pd.DataFrame, actions: np.ndarray, profile: str = ACTIVE_PROFILE) -> np.ndarray:
    actions = np.asarray(actions, dtype=int)
    rewards = reward_table(df, profile)
    oracle_values = rewards.max(axis=1)
    chosen_values = rewards[np.arange(len(df)), actions]
    return oracle_values - chosen_values

def summarize_oracle_by_profile(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    rows = []
    for profile in COST_PROFILES:
        actions, values = oracle_action_and_value(df, profile)
        rows.append({
            "split": split_name,
            "profile": profile,
            "oracle_value": float(np.mean(values)),
            "allow_rate": float(np.mean(actions == ALLOW)),
            "review_rate": float(np.mean(actions == REVIEW)),
            "block_rate": float(np.mean(actions == BLOCK)),
            "harmful_allow_rate": float(np.mean((df.label.values == 1) & (actions == ALLOW))),
            "benign_block_rate": float(np.mean((df.label.values == 0) & (actions == BLOCK))),
        })
    return pd.DataFrame(rows)

oracle_profile_table = pd.concat([
    summarize_oracle_by_profile(logged_df, "logged"),
    summarize_oracle_by_profile(eval_df, "eval"),
    summarize_oracle_by_profile(ood_df, "ood_hate"),
], ignore_index=True)

display(oracle_profile_table)


## 4. Logged bandit data with propensities

We simulate a behavior policy that resembles a legacy moderation system. It uses text heuristics and randomization.

For each example it logs:

```text
x_i, a_i, r_i, p_i
```

where `p_i = behavior_policy(a_i | x_i)`.

### Task 4

Implement the behavior policy and generate logged bandit data.

In [ ]:
def softmax(z: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    z = np.asarray(z, dtype=np.float64)
    temperature = max(float(temperature), 1e-8)
    z = z / temperature
    z = z - np.max(z)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z)

def behavior_action_probs(text: str, cfg: Config = cfg) -> np.ndarray:
    r = text_risk_score(text)

    preferences = np.array([
        2.20 - 4.00 * r,
        0.95 - 1.20 * (r - 0.55) ** 2,
        -0.75 + 3.60 * r,
    ], dtype=np.float64)

    probs = softmax(preferences, cfg.behavior_temperature)
    mix = float(np.clip(cfg.behavior_uniform_mix, 0.0, 1.0))
    probs = (1.0 - mix) * probs + mix / N_ACTIONS
    return probs / probs.sum()

def make_logged_bandit_data(df: pd.DataFrame, cfg: Config, seed: int = SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    out = df.copy().reset_index(drop=True)

    prob_matrix = np.vstack([behavior_action_probs(text, cfg) for text in out["text"].values])
    actions = np.array([rng.choice(N_ACTIONS, p=p) for p in prob_matrix], dtype=int)
    propensities = prob_matrix[np.arange(len(out)), actions]
    rewards = np.array([
        sample_observed_reward(label, sev, action, rng)
        for label, sev, action in zip(out["label"].values, out["severity"].values, actions)
    ], dtype=np.float32)

    out["logged_action"] = actions
    out["logged_action_name"] = [ACTIONS[a] for a in actions]
    out["propensity"] = propensities.astype(np.float32)
    out["reward"] = rewards
    for a, name in enumerate(ACTIONS):
        out[f"behavior_p_{name}"] = prob_matrix[:, a].astype(np.float32)
    return out

bandit_log = make_logged_bandit_data(logged_df, cfg, seed=SEED + 10)

display(bandit_log.head())
display(bandit_log["logged_action_name"].value_counts(normalize=True).rename("rate").to_frame())
display(bandit_log["propensity"].describe().to_frame())


## 5. Hashing text features
We are going to use a deteministic hashing vectorizer, which is weaker than larger models like GPT, yet is still handy with real text and supports the contextual-bandit methods.

### Task 5

Implement a hashing vectorizer.


In [ ]:
def stable_hash(token: str, dim: int) -> int:
    digest = hashlib.blake2b(token.encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, byteorder="little", signed=False) % int(dim)

def stable_sign(token: str) -> int:
    digest = hashlib.blake2b(("sign:" + token).encode("utf-8"), digest_size=1).digest()
    return 1 if (digest[0] % 2 == 0) else -1

def hashing_vectorize(texts: Iterable[str], dim: int = cfg.hash_dim, min_token_len: int = cfg.min_token_len) -> np.ndarray:
    texts = list(texts)
    X = np.zeros((len(texts), int(dim)), dtype=np.float32)

    for i, text in enumerate(texts):
        tokens = [
            t.strip()
            for t in tokenize(text)
            if len(t.strip("_@#!'")) >= int(min_token_len)
        ]
        features = list(tokens)
        features += [f"{a}__{b}" for a, b in zip(tokens, tokens[1:])]

        for feat in features:
            j = stable_hash(feat, dim)
            X[i, j] += stable_sign(feat)

    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X = X / np.maximum(norms, 1e-12)
    return X.astype(np.float32)

X_log = hashing_vectorize(bandit_log.text)
X_eval = hashing_vectorize(eval_df.text)
X_ood = hashing_vectorize(ood_df.text)

print("Feature shapes:", X_log.shape, X_eval.shape, X_ood.shape)


## 6. Neural reward model from bandit feedback

The reward model predicts one value per action:

```text
Q_theta(x) = [Q(x, allow), Q(x, review), Q(x, block)]
```

But the loss is only applied to the logged action:

```text
loss = (Q_theta(x, a_logged) - r_logged)^2
```

### Task 6

Implement the MLP reward model and chosen-action training loop.

In [ ]:
class MLPRewardModel(nn.Module):
    def __init__(self, d_in: int, hidden: int, dropout_p: float, n_actions: int = N_ACTIONS):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout_p),
        )
        self.head = nn.Linear(hidden, n_actions)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x))

    def features(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)

def clipped_inverse_propensity_weights(
    propensities: np.ndarray,
    max_weight: float = 10.0,
) -> np.ndarray:
    """Stabilized inverse-propensity weights for chosen-action reward learning.

    The reward model is trained only on logged actions. Without weighting, it
    mostly learns the behavior policy's common actions. Clipped IPW gives rare
    actions more influence while avoiding very high-variance weights.
    """
    propensities = np.maximum(np.asarray(propensities, dtype=np.float32), 1e-6)
    weights = 1.0 / propensities
    weights = np.minimum(weights, float(max_weight))
    weights = weights / max(float(weights.mean()), 1e-8)
    return weights.astype(np.float32)

def train_reward_model(
    model: nn.Module,
    X: np.ndarray,
    actions: np.ndarray,
    rewards: np.ndarray,
    cfg: Config,
    epochs: int,
    sample_weights: Optional[np.ndarray] = None,
    verbose: bool = True,
) -> nn.Module:
    model = model.to(device)

    X_t = torch.tensor(X, dtype=torch.float32)
    a_t = torch.tensor(actions, dtype=torch.long)
    r_t = torch.tensor(rewards, dtype=torch.float32)
    if sample_weights is None:
        w_t = torch.ones_like(r_t)
    else:
        w_t = torch.tensor(sample_weights, dtype=torch.float32)

    loader = DataLoader(
        TensorDataset(X_t, a_t, r_t, w_t),
        batch_size=cfg.batch_size,
        shuffle=True,
    )

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    for epoch in range(int(epochs)):
        model.train()
        total_loss = 0.0
        total_n = 0

        for xb, ab, rb, wb in loader:
            xb = xb.to(device)
            ab = ab.to(device)
            rb = rb.to(device)
            wb = wb.to(device)

            q = model(xb)
            q_logged = q.gather(1, ab.view(-1, 1)).squeeze(1)
            per_example_loss = (q_logged - rb) ** 2
            loss = (per_example_loss * wb).sum() / torch.clamp(wb.sum(), min=1e-8)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            opt.step()

            total_loss += float(loss.item()) * len(xb)
            total_n += len(xb)

        if verbose and (epoch == 0 or epoch == epochs - 1):
            print(f"reward epoch {epoch + 1:02d}/{epochs}: weighted_loss={total_loss / max(total_n, 1):.4f}")

    model.eval()
    return model

@torch.no_grad()
def predict_q(model: nn.Module, X: np.ndarray, batch_size: int = 512, mc_dropout: bool = False) -> np.ndarray:
    was_training = model.training
    if mc_dropout:
        model.train()
    else:
        model.eval()

    preds = []
    for start in range(0, len(X), batch_size):
        xb = torch.tensor(X[start:start + batch_size], dtype=torch.float32, device=device)
        preds.append(model(xb).detach().cpu().numpy())

    model.train(was_training)
    return np.vstack(preds).astype(np.float32)

reward_sample_weights = clipped_inverse_propensity_weights(
    bandit_log.propensity.values,
    max_weight=10.0,
)

display(pd.DataFrame({
    "weight_mean": [float(reward_sample_weights.mean())],
    "weight_min": [float(reward_sample_weights.min())],
    "weight_max": [float(reward_sample_weights.max())],
    "weight_p95": [float(np.quantile(reward_sample_weights, 0.95))],
}))

set_seed(SEED)
reward_model = MLPRewardModel(cfg.hash_dim, cfg.hidden, cfg.dropout_p)
reward_model = train_reward_model(
    reward_model,
    X_log,
    bandit_log.logged_action.values,
    bandit_log.reward.values,
    cfg,
    epochs=cfg.reward_epochs,
    sample_weights=reward_sample_weights,
    verbose=True,
)
display(pd.DataFrame(predict_q(reward_model, X_eval[:5]), columns=ACTIONS))

## 7. Risk model for safety and calibration

Safe exploration and calibration require an estimate of harmfulness probability.

This classifier is trained on the real dataset labels. In real deployment, labels may come from audits, human moderation, or a separate historical supervised dataset.

### Task 7

Implement a risk classifier and train it.

You will use this model later for:

- safe exploration constraints,
- calibration and reliability analysis,
- safety metrics.

In [ ]:
class RiskClassifier(nn.Module):
    def __init__(self, d_in: int, hidden: int, dropout_p: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, hidden),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(1)

def train_risk_classifier(model: nn.Module, X: np.ndarray, y: np.ndarray, cfg: Config, epochs: int) -> nn.Module:
    model = model.to(device)

    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y.astype(np.float32), dtype=torch.float32)

    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=cfg.batch_size, shuffle=True)

    n_pos = float(np.sum(y))
    n_neg = float(len(y) - n_pos)
    pos_weight = torch.tensor([n_neg / max(n_pos, 1.0)], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    for epoch in range(int(epochs)):
        model.train()
        total_loss = 0.0
        total_n = 0

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = criterion(logits, yb)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            opt.step()

            total_loss += float(loss.item()) * len(xb)
            total_n += len(xb)

        if epoch == 0 or epoch == epochs - 1:
            print(f"risk epoch {epoch + 1:02d}/{epochs}: loss={total_loss / max(total_n, 1):.4f}")

    model.eval()
    return model

@torch.no_grad()
def predict_risk(model: nn.Module, X: np.ndarray, batch_size: int = 512, mc_dropout: bool = False) -> np.ndarray:
    was_training = model.training
    if mc_dropout:
        model.train()
    else:
        model.eval()

    probs = []
    for start in range(0, len(X), batch_size):
        xb = torch.tensor(X[start:start + batch_size], dtype=torch.float32, device=device)
        logits = model(xb)
        probs.append(torch.sigmoid(logits).detach().cpu().numpy())

    model.train(was_training)
    return np.concatenate(probs).astype(np.float32)

set_seed(SEED + 100)
risk_model = RiskClassifier(cfg.hash_dim, cfg.hidden, cfg.dropout_p)
risk_model = train_risk_classifier(risk_model, X_log, bandit_log.label.values, cfg, epochs=cfg.risk_epochs)

risk_eval = predict_risk(risk_model, X_eval)
risk_ood = predict_risk(risk_model, X_ood)

display(pd.DataFrame({
    "split": ["eval", "ood_hate"],
    "mean_predicted_risk": [risk_eval.mean(), risk_ood.mean()],
    "true_label_rate": [eval_df.label.mean(), ood_df.label.mean()],
}))


## 8. Fixed policies from the reward model

Implement and evaluate:

1. Greedy reward model.
2. Epsilon-greedy reward model.
3. MC-dropout Thompson Sampling.
4. Bootstrapped ensemble Thompson Sampling.

### Task 8

Evaluate each policy on in-domain and OOD data.

In [ ]:
def greedy_from_q(q: np.ndarray) -> np.ndarray:
    return np.asarray(q).argmax(axis=1).astype(int)

def epsilon_greedy_from_q(q: np.ndarray, epsilon: float, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    greedy = greedy_from_q(q)
    explore = rng.random(len(greedy)) < float(epsilon)
    random_actions = rng.integers(0, N_ACTIONS, size=len(greedy))
    return np.where(explore, random_actions, greedy).astype(int)

def dropout_ts_policy(model: nn.Module, X: np.ndarray, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)

    q_samples = np.stack([
        predict_q(model, X, mc_dropout=True)
        for _ in range(cfg.mc_samples)
    ], axis=0)
    sample_ids = rng.integers(0, cfg.mc_samples, size=len(X))
    q_draw = q_samples[sample_ids, np.arange(len(X)), :]
    return greedy_from_q(q_draw)

def train_bootstrap_ensemble(
    X: np.ndarray,
    actions: np.ndarray,
    rewards: np.ndarray,
    cfg: Config,
    k: int,
    seed: int,
    sample_weights: Optional[np.ndarray] = None,
) -> List[nn.Module]:
    rng = np.random.default_rng(seed)
    models = []

    for m in range(int(k)):
        set_seed(seed + m)
        idx = rng.integers(0, len(X), size=len(X))
        model = MLPRewardModel(cfg.hash_dim, cfg.hidden, cfg.dropout_p)
        model = train_reward_model(
            model,
            X[idx],
            actions[idx],
            rewards[idx],
            cfg,
            epochs=cfg.ensemble_epochs,
            sample_weights=None if sample_weights is None else sample_weights[idx],
            verbose=False,
        )
        models.append(model)
        print(f"trained bootstrap model {m + 1}/{k}")

    return models

def ensemble_ts_policy(models: List[nn.Module], X: np.ndarray, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    q_stack = np.stack([predict_q(model, X) for model in models], axis=0)
    model_ids = rng.integers(0, len(models), size=len(X))
    q_draw = q_stack[model_ids, np.arange(len(X)), :]
    return greedy_from_q(q_draw)

def evaluate_action_vector(df: pd.DataFrame, actions: np.ndarray, name: str, profile: str = ACTIVE_PROFILE) -> Dict[str, float]:
    actions = np.asarray(actions, dtype=int)
    rewards = reward_table(df, profile)
    chosen_values = rewards[np.arange(len(df)), actions]
    regrets = expected_regret(df, actions, profile)
    oracle_actions, oracle_values = oracle_action_and_value(df, profile)

    labels = df["label"].values.astype(int)
    severities = df["severity"].values.astype(float)
    harmful_allowed = (labels == 1) & (actions == ALLOW)

    return {
        "policy": name,
        "value": float(np.mean(chosen_values)),
        "oracle_value": float(np.mean(oracle_values)),
        "avg_regret": float(np.mean(regrets)),
        "allow_rate": float(np.mean(actions == ALLOW)),
        "review_rate": float(np.mean(actions == REVIEW)),
        "block_rate": float(np.mean(actions == BLOCK)),
        "harmful_allow_rate": float(np.mean(harmful_allowed)),
        "benign_block_rate": float(np.mean((labels == 0) & (actions == BLOCK))),
        "oracle_match_rate": float(np.mean(actions == oracle_actions)),
        "mean_severity_harm_allowed": float(np.mean(severities[harmful_allowed])) if np.any(harmful_allowed) else 0.0,
    }

ensemble = train_bootstrap_ensemble(
    X_log,
    bandit_log.logged_action.values,
    bandit_log.reward.values,
    cfg,
    k=cfg.ensemble_size,
    seed=SEED + 200,
    sample_weights=reward_sample_weights,
)

def evaluate_fixed_policies(df: pd.DataFrame, X: np.ndarray, title: str) -> pd.DataFrame:
    q = predict_q(reward_model, X)
    policies = {
        "oracle": oracle_action_and_value(df)[0],
        "greedy_mlp": greedy_from_q(q),
        f"epsilon_{cfg.epsilon:.2f}_mlp": epsilon_greedy_from_q(q, cfg.epsilon, SEED + 1),
        "dropout_ts_mlp": dropout_ts_policy(reward_model, X, SEED + 2),
        "ensemble_ts_mlp": ensemble_ts_policy(ensemble, X, SEED + 3),
    }
    rows = [evaluate_action_vector(df, acts, name) for name, acts in policies.items()]
    out = pd.DataFrame(rows).sort_values("avg_regret")
    print(title)
    display(out)
    return out

eval_results = evaluate_fixed_policies(eval_df, X_eval, "In-domain fixed-policy evaluation")
ood_results = evaluate_fixed_policies(ood_df, X_ood, "OOD hate fixed-policy evaluation")


## 9. Safe exploration constraints


Naive exploration can randomly `allow` harmful content or randomly `block` benign content. In a real moderation system, exploration must be constrained.

We define a risk-aware action mask:

- if predicted harmfulness is below `unsafe_threshold`, all actions are allowed;
- if predicted harmfulness is above `unsafe_threshold`, `allow` is disallowed;
- if predicted harmfulness is above `extreme_unsafe_threshold`, only `review` and `block` are allowed, and `review` is preferred unless `block` has enough predicted advantage.

### Task 9

Implement safe action masking and safe versions of epsilon-greedy and Thompson Sampling.


In [ ]:
def safe_action_mask(
    risk_probs: np.ndarray,
    unsafe_threshold: float = cfg.unsafe_threshold,
    extreme_threshold: float = cfg.extreme_unsafe_threshold,
) -> np.ndarray:
    risk_probs = np.asarray(risk_probs, dtype=np.float32)
    mask = np.ones((len(risk_probs), N_ACTIONS), dtype=bool)

    unsafe = risk_probs >= float(unsafe_threshold)
    mask[unsafe, ALLOW] = False

    # With three actions, the extreme case still allows review and block. The
    # "prefer review unless block is clearly better" rule is enforced by the
    # safe policy functions below using cfg.safety_margin.
    extreme = risk_probs >= float(extreme_threshold)
    mask[extreme, :] = False
    mask[extreme, REVIEW] = True
    mask[extreme, BLOCK] = True

    return mask

def masked_argmax(q: np.ndarray, mask: np.ndarray) -> np.ndarray:
    q = np.asarray(q, dtype=np.float32)
    mask = np.asarray(mask, dtype=bool)
    masked_q = np.where(mask, q, -np.inf)
    return masked_q.argmax(axis=1).astype(int)

def _apply_extreme_review_margin(actions: np.ndarray, q: np.ndarray, risk_probs: np.ndarray) -> np.ndarray:
    actions = np.asarray(actions, dtype=int).copy()
    extreme = np.asarray(risk_probs) >= cfg.extreme_unsafe_threshold
    block_has_margin = q[:, BLOCK] >= q[:, REVIEW] + cfg.safety_margin
    actions[extreme & ~block_has_margin] = REVIEW
    actions[extreme & block_has_margin] = BLOCK
    return actions

def safe_greedy_policy(q: np.ndarray, risk_probs: np.ndarray) -> np.ndarray:
    mask = safe_action_mask(risk_probs)
    actions = masked_argmax(q, mask)
    return _apply_extreme_review_margin(actions, q, risk_probs)

def safe_epsilon_greedy_policy(q: np.ndarray, risk_probs: np.ndarray, epsilon: float, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    mask = safe_action_mask(risk_probs)
    greedy = safe_greedy_policy(q, risk_probs)
    actions = greedy.copy()

    for i in range(len(actions)):
        if rng.random() < float(epsilon):
            allowed = np.flatnonzero(mask[i])
            actions[i] = int(rng.choice(allowed))

    return _apply_extreme_review_margin(actions, q, risk_probs).astype(int)

def safe_dropout_ts_policy(model: nn.Module, X: np.ndarray, risk_probs: np.ndarray, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    q_samples = np.stack([
        predict_q(model, X, mc_dropout=True)
        for _ in range(cfg.mc_samples)
    ], axis=0)
    sample_ids = rng.integers(0, cfg.mc_samples, size=len(X))
    q_draw = q_samples[sample_ids, np.arange(len(X)), :]
    return safe_greedy_policy(q_draw, risk_probs)

def safe_ensemble_ts_policy(models: List[nn.Module], X: np.ndarray, risk_probs: np.ndarray, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    q_stack = np.stack([predict_q(model, X) for model in models], axis=0)
    model_ids = rng.integers(0, len(models), size=len(X))
    q_draw = q_stack[model_ids, np.arange(len(X)), :]
    return safe_greedy_policy(q_draw, risk_probs)

def evaluate_safe_policies(df: pd.DataFrame, X: np.ndarray, risk_probs: np.ndarray, title: str) -> pd.DataFrame:
    q = predict_q(reward_model, X)
    policies = {
        "safe_greedy_mlp": safe_greedy_policy(q, risk_probs),
        f"safe_epsilon_{cfg.epsilon:.2f}_mlp": safe_epsilon_greedy_policy(q, risk_probs, cfg.epsilon, SEED + 4),
        "safe_dropout_ts_mlp": safe_dropout_ts_policy(reward_model, X, risk_probs, SEED + 5),
        "safe_ensemble_ts_mlp": safe_ensemble_ts_policy(ensemble, X, risk_probs, SEED + 6),
    }
    rows = [evaluate_action_vector(df, acts, name) for name, acts in policies.items()]
    out = pd.DataFrame(rows).sort_values("avg_regret")
    print(title)
    display(out)
    return out

safe_eval_results = evaluate_safe_policies(eval_df, X_eval, risk_eval, "Mandatory safe exploration: in-domain")
safe_ood_results = evaluate_safe_policies(ood_df, X_ood, risk_ood, "Mandatory safe exploration: OOD hate")


## 10. Online LinUCB

LinUCB is a classic contextual bandit method. It maintains a separate ridge regression model for each action and adds an uncertainty bonus.

### Task 10

Implement raw-feature LinUCB and evaluate cumulative regret on the in-domain stream.


In [ ]:
class LinUCB:
    def __init__(self, d: int, n_actions: int = N_ACTIONS, alpha: float = 0.5, l2: float = 1.0):
        self.d = int(d)
        self.n_actions = int(n_actions)
        self.alpha = float(alpha)
        self.l2 = float(l2)

        self.A_inv = np.stack([
            np.eye(self.d, dtype=np.float64) / self.l2
            for _ in range(self.n_actions)
        ], axis=0)
        self.b = np.zeros((self.n_actions, self.d), dtype=np.float64)

    def select(self, x: np.ndarray, allowed_actions: Optional[np.ndarray] = None) -> int:
        x = np.asarray(x, dtype=np.float64)
        if allowed_actions is None:
            allowed_actions = np.ones(self.n_actions, dtype=bool)
        allowed_actions = np.asarray(allowed_actions, dtype=bool)

        scores = np.full(self.n_actions, -np.inf, dtype=np.float64)
        for a in np.flatnonzero(allowed_actions):
            A_inv_x = self.A_inv[a] @ x
            theta = self.A_inv[a] @ self.b[a]
            mean = float(theta @ x)
            uncertainty = math.sqrt(max(float(x @ A_inv_x), 0.0))
            scores[a] = mean + self.alpha * uncertainty

        return int(np.argmax(scores))

    def update(self, x: np.ndarray, action: int, reward: float):
        x = np.asarray(x, dtype=np.float64)
        action = int(action)

        A_inv_x = self.A_inv[action] @ x
        denom = 1.0 + float(x @ A_inv_x)
        self.A_inv[action] -= np.outer(A_inv_x, A_inv_x) / max(denom, 1e-12)
        self.b[action] += float(reward) * x

def run_online_linucb(
    df: pd.DataFrame,
    X: np.ndarray,
    alpha: float,
    l2: float,
    name: str,
    risk_probs: Optional[np.ndarray] = None,
    seed: int = SEED,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    bandit = LinUCB(d=X.shape[1], n_actions=N_ACTIONS, alpha=alpha, l2=l2)

    rewards = reward_table(df)
    oracle_values = rewards.max(axis=1)
    order = rng.permutation(len(df))
    masks = safe_action_mask(risk_probs) if risk_probs is not None else None

    rows = []
    cum_regret = 0.0

    for t, i in enumerate(order, start=1):
        allowed = masks[i] if masks is not None else None
        action = bandit.select(X[i], allowed_actions=allowed)
        observed_reward = sample_observed_reward(
            int(df.label.iloc[i]),
            float(df.severity.iloc[i]),
            action,
            rng,
        )

        expected_value = float(rewards[i, action])
        regret = float(oracle_values[i] - expected_value)
        cum_regret += regret

        bandit.update(X[i], action, observed_reward)

        rows.append({
            "t": t,
            "row_id": int(i),
            "policy": name,
            "action": action,
            "action_name": ACTIONS[action],
            "reward": float(observed_reward),
            "expected_value": expected_value,
            "oracle_value": float(oracle_values[i]),
            "regret": regret,
            "cum_regret": cum_regret,
        })

    return pd.DataFrame(rows)

linucb_trace = run_online_linucb(eval_df, X_eval, cfg.linucb_alpha, cfg.linucb_l2, "linucb_raw")
safe_linucb_trace = run_online_linucb(eval_df, X_eval, cfg.linucb_alpha, cfg.linucb_l2, "safe_linucb_raw", risk_probs=risk_eval)
display(pd.concat([linucb_trace.tail(1), safe_linucb_trace.tail(1)], ignore_index=True))


## 11. NeuralUCB-style last-layer LinUCB


The neural reward model learned a representation. Instead of running LinUCB on raw hashed features, extract the penultimate representation and run LinUCB on those features.

This approximates a NeuralUCB-style method:

```text
phi(x) = neural_encoder(x)
LinUCB(phi(x))
```

### Task 11

Implement feature extraction and NeuralUCB-style evaluation.

In [ ]:
@torch.no_grad()
def extract_neural_features(model: MLPRewardModel, X: np.ndarray, batch_size: int = 512) -> np.ndarray:
    was_training = model.training
    model.eval()

    feats = []
    for start in range(0, len(X), batch_size):
        xb = torch.tensor(X[start:start + batch_size], dtype=torch.float32, device=device)
        h = model.features(xb)
        feats.append(h.detach().cpu().numpy())

    model.train(was_training)

    H = np.vstack(feats).astype(np.float32)
    norms = np.linalg.norm(H, axis=1, keepdims=True)
    return H / np.maximum(norms, 1e-12)

H_log = extract_neural_features(reward_model, X_log)
H_eval = extract_neural_features(reward_model, X_eval)
H_ood = extract_neural_features(reward_model, X_ood)

print("Neural feature shapes:", H_log.shape, H_eval.shape, H_ood.shape)

neuralucb_trace = run_online_linucb(
    eval_df, H_eval,
    alpha=cfg.neuralucb_alpha,
    l2=cfg.linucb_l2,
    name="neuralucb_last_layer",
    risk_probs=None,
    seed=SEED + 300,
)

safe_neuralucb_trace = run_online_linucb(
    eval_df, H_eval,
    alpha=cfg.neuralucb_alpha,
    l2=cfg.linucb_l2,
    name="safe_neuralucb_last_layer",
    risk_probs=risk_eval,
    seed=SEED + 301,
)

display(pd.concat([
    linucb_trace.tail(1),
    neuralucb_trace.tail(1),
    safe_neuralucb_trace.tail(1),
], ignore_index=True))


## 12. Offline policy evaluation: IPS, SNIPS, Direct Method, Doubly Robust

Before deploying a policy, estimate its value from logs.

You have:

```text
x_i, a_i, r_i, p_i
```

where `p_i` is the behavior propensity.

### Task 12

Implement:

- IPS
- SNIPS
- Direct Method
- Doubly Robust

Compare them to the simulator's true value on the logged contexts.

In [ ]:
def deterministic_policy_probs(actions: np.ndarray, n_actions: int = N_ACTIONS) -> np.ndarray:
    actions = np.asarray(actions, dtype=int)
    P = np.zeros((len(actions), n_actions), dtype=np.float32)
    P[np.arange(len(actions)), actions] = 1.0
    return P

def ips_value(log: pd.DataFrame, target_probs: np.ndarray) -> float:
    logged_actions = log["logged_action"].values.astype(int)
    prop = np.maximum(log["propensity"].values.astype(float), 1e-12)
    rewards = log["reward"].values.astype(float)
    weights = target_probs[np.arange(len(log)), logged_actions] / prop
    return float(np.mean(weights * rewards))

def snips_value(log: pd.DataFrame, target_probs: np.ndarray) -> float:
    logged_actions = log["logged_action"].values.astype(int)
    prop = np.maximum(log["propensity"].values.astype(float), 1e-12)
    rewards = log["reward"].values.astype(float)
    weights = target_probs[np.arange(len(log)), logged_actions] / prop
    denom = np.sum(weights)
    if denom <= 1e-12:
        return float("nan")
    return float(np.sum(weights * rewards) / denom)

def direct_method_value(q_hat: np.ndarray, target_probs: np.ndarray) -> float:
    return float(np.mean(np.sum(target_probs * q_hat, axis=1)))

def doubly_robust_value(log: pd.DataFrame, q_hat: np.ndarray, target_probs: np.ndarray) -> float:
    logged_actions = log["logged_action"].values.astype(int)
    prop = np.maximum(log["propensity"].values.astype(float), 1e-12)
    rewards = log["reward"].values.astype(float)

    dm_part = np.sum(target_probs * q_hat, axis=1)
    q_logged = q_hat[np.arange(len(log)), logged_actions]
    weights = target_probs[np.arange(len(log)), logged_actions] / prop
    dr = dm_part + weights * (rewards - q_logged)
    return float(np.mean(dr))

def true_value_on_contexts(df: pd.DataFrame, actions: np.ndarray, profile: str = ACTIVE_PROFILE) -> float:
    actions = np.asarray(actions, dtype=int)
    rewards = reward_table(df, profile)
    return float(np.mean(rewards[np.arange(len(df)), actions]))

q_log = predict_q(reward_model, X_log)
risk_log = predict_risk(risk_model, X_log)

target_actions = {
    "greedy_mlp": greedy_from_q(q_log),
    "safe_greedy_mlp": safe_greedy_policy(q_log, risk_log),
    "dropout_ts_mlp": dropout_ts_policy(reward_model, X_log, SEED + 12),
    "safe_dropout_ts_mlp": safe_dropout_ts_policy(reward_model, X_log, risk_log, SEED + 13),
    "ensemble_ts_mlp": ensemble_ts_policy(ensemble, X_log, SEED + 14),
}

ope_rows = []
for name, acts in target_actions.items():
    P = deterministic_policy_probs(acts)
    ope_rows.append({
        "policy": name,
        "true_value": true_value_on_contexts(bandit_log, acts),
        "IPS": ips_value(bandit_log, P),
        "SNIPS": snips_value(bandit_log, P),
        "DM": direct_method_value(q_log, P),
        "DR": doubly_robust_value(bandit_log, q_log, P),
        "match_rate": float(np.mean(bandit_log.logged_action.values == acts)),
    })

ope_table = pd.DataFrame(ope_rows)
display(ope_table)


## 13. Calibration and reliability analysis

This section is now required.

A moderation system should not only rank risks; it should produce probabilities that mean something. If the model predicts 0.80 harmfulness for 1000 examples, roughly 800 should be harmful for good calibration.

### Task 13

Implement:

- calibration bins,
- Expected Calibration Error,
- reliability plot,
- risk-threshold analysis.

Evaluate calibration on both in-domain and OOD data.

In [ ]:
def calibration_table(y_true: np.ndarray, p_pred: np.ndarray, n_bins: int = 10) -> pd.DataFrame:
    y_true = np.asarray(y_true, dtype=float)
    p_pred = np.clip(np.asarray(p_pred, dtype=float), 0.0, 1.0)
    rows = []

    for b in range(n_bins):
        lo = b / n_bins
        hi = (b + 1) / n_bins
        if b == n_bins - 1:
            in_bin = (p_pred >= lo) & (p_pred <= hi)
        else:
            in_bin = (p_pred >= lo) & (p_pred < hi)

        count = int(in_bin.sum())
        if count > 0:
            avg_pred = float(p_pred[in_bin].mean())
            empirical = float(y_true[in_bin].mean())
            gap = abs(avg_pred - empirical)
        else:
            avg_pred = np.nan
            empirical = np.nan
            gap = 0.0

        rows.append({
            "bin": b,
            "low": lo,
            "high": hi,
            "count": count,
            "weight": count / max(len(y_true), 1),
            "avg_pred": avg_pred,
            "empirical_rate": empirical,
            "abs_gap": float(gap),
        })

    return pd.DataFrame(rows)

def expected_calibration_error(y_true: np.ndarray, p_pred: np.ndarray, n_bins: int = 10) -> float:
    tab = calibration_table(y_true, p_pred, n_bins)
    return float((tab["weight"] * tab["abs_gap"]).sum())

def plot_reliability(y_true: np.ndarray, p_pred: np.ndarray, title: str, n_bins: int = 10):
    tab = calibration_table(y_true, p_pred, n_bins)

    plot_tab = tab[tab["count"] > 0]
    plt.figure(figsize=(5, 4))
    plt.plot([0, 1], [0, 1], linestyle="--", label="perfect calibration")
    plt.plot(plot_tab["avg_pred"], plot_tab["empirical_rate"], marker="o", label="model")
    plt.xlabel("Mean predicted harmfulness")
    plt.ylabel("Empirical harmful rate")
    plt.title(f"{title}\nECE={expected_calibration_error(y_true, p_pred, n_bins):.3f}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    return tab

eval_cal = plot_reliability(eval_df.label.values, risk_eval, "Mandatory calibration: in-domain")
ood_cal = plot_reliability(ood_df.label.values, risk_ood, "Mandatory calibration: OOD hate")

display(pd.DataFrame({
    "split": ["eval", "ood_hate"],
    "ECE_10_bins": [
        expected_calibration_error(eval_df.label.values, risk_eval, 10),
        expected_calibration_error(ood_df.label.values, risk_ood, 10),
    ],
    "mean_pred": [risk_eval.mean(), risk_ood.mean()],
    "empirical_rate": [eval_df.label.mean(), ood_df.label.mean()],
}))

def risk_threshold_analysis(df: pd.DataFrame, risk_probs: np.ndarray, thresholds: List[float]) -> pd.DataFrame:
    y = df["label"].values.astype(int)
    severities = df["severity"].values.astype(float)
    rows = []

    for th in thresholds:
        flagged = np.asarray(risk_probs) >= float(th)

        tp = int(np.sum(flagged & (y == 1)))
        fp = int(np.sum(flagged & (y == 0)))
        fn = int(np.sum((~flagged) & (y == 1)))
        tn = int(np.sum((~flagged) & (y == 0)))

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        fpr = fp / max(fp + tn, 1)
        fnr = fn / max(tp + fn, 1)

        rows.append({
            "threshold": float(th),
            "flag_rate": float(flagged.mean()),
            "precision": float(precision),
            "recall": float(recall),
            "false_positive_rate": float(fpr),
            "false_negative_rate": float(fnr),
            "mean_severity_flagged": float(severities[flagged].mean()) if np.any(flagged) else np.nan,
        })

    return pd.DataFrame(rows)

thresholds = [0.2, 0.35, 0.5, 0.65, 0.8]
display(risk_threshold_analysis(eval_df, risk_eval, thresholds))
display(risk_threshold_analysis(ood_df, risk_ood, thresholds))


## 14. Final comparison and required written analysis

Now compare:

- unconstrained policies,
- safe policies,
- raw LinUCB,
- safe raw LinUCB,
- NeuralUCB-style LinUCB,
- safe NeuralUCB-style LinUCB,
- in-domain vs OOD behavior,
- calibration degradation.

### Task 14

Produce a final table and write a short explanation (here in the notebook) about it.

In [ ]:
def policy_degradation_table(eval_results: pd.DataFrame, ood_results: pd.DataFrame) -> pd.DataFrame:
    left = eval_results.copy().add_prefix("eval_")
    right = ood_results.copy().add_prefix("ood_")

    merged = left.merge(
        right,
        left_on="eval_policy",
        right_on="ood_policy",
        how="inner",
    )
    merged["policy"] = merged["eval_policy"]
    merged["value_drop_eval_minus_ood"] = merged["eval_value"] - merged["ood_value"]
    merged["regret_increase_ood_minus_eval"] = merged["ood_avg_regret"] - merged["eval_avg_regret"]
    merged["harmful_allow_increase_ood_minus_eval"] = (
        merged["ood_harmful_allow_rate"] - merged["eval_harmful_allow_rate"]
    )

    keep = [
        "policy",
        "eval_value",
        "ood_value",
        "value_drop_eval_minus_ood",
        "eval_avg_regret",
        "ood_avg_regret",
        "regret_increase_ood_minus_eval",
        "eval_harmful_allow_rate",
        "ood_harmful_allow_rate",
        "harmful_allow_increase_ood_minus_eval",
        "eval_benign_block_rate",
        "ood_benign_block_rate",
    ]
    return merged[keep].sort_values("ood_avg_regret")

unconstrained_degradation = policy_degradation_table(eval_results, ood_results)
safe_degradation = policy_degradation_table(safe_eval_results, safe_ood_results)

display(unconstrained_degradation)
display(safe_degradation)

online_summary = pd.DataFrame([
    {
        "policy": "linucb_raw",
        "cum_regret": float(linucb_trace.cum_regret.iloc[-1]),
        "avg_regret": float(linucb_trace.regret.mean()),
    },
    {
        "policy": "safe_linucb_raw",
        "cum_regret": float(safe_linucb_trace.cum_regret.iloc[-1]),
        "avg_regret": float(safe_linucb_trace.regret.mean()),
    },
    {
        "policy": "neuralucb_last_layer",
        "cum_regret": float(neuralucb_trace.cum_regret.iloc[-1]),
        "avg_regret": float(neuralucb_trace.regret.mean()),
    },
    {
        "policy": "safe_neuralucb_last_layer",
        "cum_regret": float(safe_neuralucb_trace.cum_regret.iloc[-1]),
        "avg_regret": float(safe_neuralucb_trace.regret.mean()),
    },
])
display(online_summary.sort_values("avg_regret"))

plt.figure(figsize=(7, 4))
for trace in [linucb_trace, safe_linucb_trace, neuralucb_trace, safe_neuralucb_trace]:
    plt.plot(trace["t"], trace["cum_regret"], label=trace["policy"].iloc[0])
plt.xlabel("Online step")
plt.ylabel("Cumulative expected regret")
plt.title("Online LinUCB regret comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

combined_eval = pd.concat([
    eval_results.assign(group="unconstrained"),
    safe_eval_results.assign(group="safe"),
], ignore_index=True)
combined_ood = pd.concat([
    ood_results.assign(group="unconstrained"),
    safe_ood_results.assign(group="safe"),
], ignore_index=True)

summary_plot = pd.concat([
    combined_eval[["policy", "avg_regret"]].assign(split="eval"),
    combined_ood[["policy", "avg_regret"]].assign(split="ood_hate"),
], ignore_index=True)

pivot = summary_plot.pivot_table(index="policy", columns="split", values="avg_regret")
display(pivot.sort_values("ood_hate"))

ax = pivot.sort_values("ood_hate").plot(kind="bar", figsize=(8, 4))
ax.set_ylabel("Average expected regret")
ax.set_title("Fixed-policy regret: in-domain vs OOD")
ax.tick_params(axis="x", rotation=45)
plt.xticks(ha="right")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

try:
    from IPython.display import Markdown

    learned_eval = combined_eval[combined_eval["policy"] != "oracle"].copy()
    learned_ood = combined_ood[combined_ood["policy"] != "oracle"].copy()

    best_eval = learned_eval.sort_values("avg_regret").iloc[0]
    best_ood = learned_ood.sort_values("avg_regret").iloc[0]
    best_online = online_summary.sort_values("avg_regret").iloc[0]

    ece_eval = expected_calibration_error(eval_df.label.values, risk_eval, 10)
    ece_ood = expected_calibration_error(ood_df.label.values, risk_ood, 10)

    safe_best_eval = safe_eval_results.sort_values("avg_regret").iloc[0]
    unsafe_best_eval = eval_results[eval_results.policy != "oracle"].sort_values("avg_regret").iloc[0]
    safe_harm_delta = safe_best_eval.harmful_allow_rate - unsafe_best_eval.harmful_allow_rate

    best_ope_true = ope_table.sort_values("true_value", ascending=False).iloc[0]
    mean_ope_overlap = float(ope_table["match_rate"].mean())
    mean_ips_error = float(np.mean(np.abs(ope_table["IPS"] - ope_table["true_value"])))
    mean_dr_error = float(np.mean(np.abs(ope_table["DR"] - ope_table["true_value"])))

except Exception:
    print("Final report generated above; IPython Markdown display is unavailable in this environment.")


The oracle is only an upper-bound reference, not a real deployable policy.

The learned policies perform worse than the oracle because they are trained from limited logged feedback, where some actions are observed much more often than others. Safe policies can reduce harmful decisions, but they may also block or review more content because the risk model is imperfect.

NeuralUCB performs better than basic LinUCB because it uses stronger learned text representations. Offline policy evaluation is noisy when the learned policy chooses actions that were rarely chosen in the logged data.

Overall, the results show that offline contextual bandits are useful for moderation, but they depend heavily on good logging coverage, reliable risk estimates, and proper calibration, especially on OOD data.


## Grading rubric

| Section | Points |
|---|---:|
| Real dataset loading, EDA, severity critique | 10 |
| Reward, oracle, regret | 10 |
| cost-sensitivity analysis | 10 |
| Logged bandit feedback and propensities | 10 |
| Reward model from chosen-action feedback | 15 |
| Fixed policies and uncertainty policies | 10 |
| safe exploration | 15 |
| Raw LinUCB | 10 |
| NeuralUCB-style LinUCB | 15 |
| Offline policy evaluation | 15 |
| calibration analysis | 15 |
| OOD stress analysis | 10 |
| Final safety/product discussion | 15 |


Total: 160 points.


PDF Question Answers

1. Dataset

(a) Binary labels are limited because they only say harmful or safe. They do not capture severity, intent, sarcasm, context, or different costs of allow, review, and block.

(b) tweet_eval/hate is a reasonable OOD test because it has a similar short social-media style, but it focuses on hate speech instead of general offensive content. It is imperfect because it is still a curated binary dataset and does not fully represent real moderation traffic.

2. Severity Proxy

(a) A severity proxy is dangerous as ground truth because it is only a heuristic. It may encode bias, overreact to certain words, and miss implicit harmful content.

(b) It mainly overemphasizes surface signals such as risky words, capital letters, punctuation, and hashtags, while ignoring deeper context and intent.

3. Reward, Oracle, Regret, and Cost Sensitivity

(a) The high-safety profile is most likely to over-block because it gives a high cost to allowing harmful content, so the policy becomes more conservative.

(b) This is a bandit problem because the system chooses actions and only observes feedback for the chosen action. It is not just predicting a label.

(c) The oracle action distribution changes because different cost profiles make different actions more or less valuable. It also changes when the harmfulness distribution changes across datasets.

4. Logged Bandit Data

(a) Low propensities make offline evaluation unreliable because rare actions get very large importance weights. This makes IPS and DR estimates noisy and sometimes misleading.

5. Risk Model for Safety and Calibration

(a) It is acceptable in this experiment because the risk model uses labels to estimate harmfulness, while the reward model learns from logged action rewards. In a real system, this must be handled carefully because labels can be biased, delayed, or missing.

(b) If labels are delayed or missing, training becomes harder. The system would need more conservative decisions, human review, delayed-feedback handling, and calibration checks.

6. Fixed Policies from the Reward Model

(a) Yes, uncertainty can help because it encourages more cautious exploration instead of always choosing the greedy predicted action.

(b) The policies with more exploration or stronger safety constraints usually have the highest block rate.

(c) The policy with the worst OOD degradation is the one whose performance drops most when moving from in-domain data to hate data, showing poor generalization.

7. Safe Exploration Constraints

(a) Ethical exploration should avoid randomly allowing high-risk content. Uncertain or risky cases should go to review or block, while exploration should only happen among reasonably safe actions.

8. Online LinUCB

(a) LinUCB can be more stable early because it is simpler and has explicit uncertainty estimates. Neural Thompson sampling can be noisier when there is little data.

9. Offline Policy Evaluation

(a) IPS is usually the most variable estimator because it depends directly on inverse propensity weights.

(b) The Direct Method is most biased when the reward model is poor because it relies completely on predicted rewards.

(c) Very low behavior propensities make evaluation unstable. Importance weights become too large, and clipping reduces variance but adds bias.

10. Calibration

(a) Miscalibration makes safe exploration unreliable. Underestimated risk can allow harmful content, while overestimated risk can cause unnecessary review or blocking of benign content.
